# 💬 Week 6: Analisis Sentimen (Sentiment Analysis)

## Tujuan Pembelajaran:
1. Memahami konsep **Analisis Sentimen** — menentukan apakah suatu teks bersifat positif, negatif, atau netral
2. Menerapkan **TextBlob** untuk mengukur polaritas teks secara otomatis
3. Mengkategorikan ulasan Honest Review berdasarkan sentimen
4. Menganalisis **tren jumlah ulasan** dan **tren sentimen** dari waktu ke waktu
5. Membuat **Word Cloud** dari keseluruhan teks ulasan

---
> 📂 **Dataset**: Honest Review — file `df_honest_reviews_id.csv`, kolom `content` (teks ulasan) dan `at` (tanggal)
> File ini berada di folder `Week 2/` dalam workspace yang sama.


## 1) Install Library

Install library yang dibutuhkan:
- `textblob` — library NLP ringan untuk analisis sentimen
- `wordcloud` — untuk membuat visualisasi Word Cloud


In [ ]:
%pip install textblob wordcloud pandas matplotlib


## 2) Import Library

Import modul yang diperlukan untuk analisis sentimen dan visualisasi.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob
from wordcloud import WordCloud

print("Library berhasil diimport!")

## 3) Load Dataset

Load dataset Honest Review dari `df_honest_reviews_id.csv`. Kolom `content` berisi teks ulasan, dan kolom `at` berisi tanggal ulasan ditulis.


In [ ]:
file_path = '../Week 2/df_honest_reviews_id.csv'
df = pd.read_csv(file_path)

print(f"Jumlah data: {len(df)}")
print(f"Kolom: {list(df.columns)}")
df.head()


## 4) Fungsi Analisis Sentimen

**TextBlob** menganalisis sentimen teks berbasis leksikon Inggris. Nilai polaritas:
- **> 0** = Positif
- **= 0** = Netral  
- **< 0** = Negatif

Karena teks adalah bahasa Indonesia, TextBlob akan menganalisis berdasarkan kata-kata yang dikenalinya (kata serapan, istilah umum, dll). Untuk akurasi lebih tinggi pada teks Indonesia, disarankan menggunakan model berbasis BERT (lihat notebook NER.ipynb).

In [ ]:
def analyze_sentiment_textblob(text):
    """
    Menganalisis polaritas sentimen teks menggunakan TextBlob.
    Return nilai float antara -1.0 (sangat negatif) dan 1.0 (sangat positif).
    """
    analysis = TextBlob(str(text))
    return analysis.sentiment.polarity


def categorize_sentiment(polarity):
    """
    Mengkategorikan nilai polaritas menjadi label teks.
    """
    if polarity > 0:
        return 'Positif'
    elif polarity < 0:
        return 'Negatif'
    else:
        return 'Netral'


# Contoh penggunaan
contoh_teks = "Aplikasi ini sangat bagus dan mudah digunakan untuk belanja."
polarity = analyze_sentiment_textblob(contoh_teks)
print(f"Teks   : '{contoh_teks}'")
print(f"Polarity: {polarity:.4f}  →  {categorize_sentiment(polarity)}")


## 5) Terapkan Analisis Sentimen ke Seluruh Dataset

Terapkan fungsi sentimen ke kolom `content` untuk mendapatkan nilai polaritas dan kategori sentimen untuk setiap ulasan.


In [ ]:
# Hitung polaritas dan kategori sentimen
df['sentiment_textblob'] = df['content'].apply(analyze_sentiment_textblob)
df['sentiment_category'] = df['sentiment_textblob'].apply(categorize_sentiment)

print("5 data pertama dengan hasil analisis sentimen:")
df[['content', 'sentiment_textblob', 'sentiment_category']].head()


## 6) Distribusi Nilai Sentimen

Visualisasikan distribusi nilai polaritas untuk melihat sebaran sentimen secara keseluruhan.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df['sentiment_textblob'], bins=20, color='steelblue', alpha=0.8, edgecolor='white')
plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Netral (0)')
plt.title('Distribusi Nilai Polaritas Sentimen (TextBlob)', fontsize=13)
plt.xlabel('Nilai Polaritas  (-1 = Sangat Negatif, +1 = Sangat Positif)')
plt.ylabel('Frekuensi')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7) Kategorisasi Sentimen

Hitung jumlah ulasan per kategori sentimen (Positif, Netral, Negatif) dan tampilkan dalam bar plot.


In [ ]:
sentiment_counts = df['sentiment_category'].value_counts()

print("Distribusi Kategori Sentimen:")
print(sentiment_counts)
print(f"\nTotal ulasan: {len(df)}")

# Bar plot
color_map = {'Positif': 'mediumseagreen', 'Netral': 'lightskyblue', 'Negatif': 'tomato'}
bar_colors = [color_map.get(cat, 'gray') for cat in sentiment_counts.index]

plt.figure(figsize=(6, 4))
sentiment_counts.plot(kind='bar', color=bar_colors, alpha=0.85, edgecolor='white')
plt.title('Kategorisasi Sentimen Ulasan Honest Review (TextBlob)', fontsize=13)
plt.xlabel('Kategori Sentimen')
plt.ylabel('Jumlah Ulasan')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 8) Tren Jumlah Ulasan dari Waktu ke Waktu

Analisis seberapa banyak ulasan Honest Review yang ditulis setiap harinya untuk melihat tren aktivitas pengguna.


In [ ]:
# Konversi kolom tanggal ke format datetime
df['at'] = pd.to_datetime(df['at'])

# Hitung jumlah ulasan per tanggal
trends = df.groupby('at').size()

plt.figure(figsize=(12, 5))
plt.plot(trends.index, trends.values, marker='o', linestyle='-', color='steelblue', markersize=4)
plt.title('Tren Jumlah Ulasan Honest Review dari Waktu ke Waktu', fontsize=13)
plt.xlabel('Tanggal')
plt.ylabel('Jumlah Ulasan')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9) Tren Sentimen dari Waktu ke Waktu

Hitung rata-rata polaritas sentimen per hari untuk melihat bagaimana sentimen ulasan Honest Review berubah dari waktu ke waktu.


In [ ]:
# Rata-rata sentimen per tanggal
sentiment_trends = df.groupby('at')['sentiment_textblob'].mean()

plt.figure(figsize=(12, 5))
plt.plot(sentiment_trends.index, sentiment_trends.values, marker='o', linestyle='-',
         color='darkorange', markersize=4)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.7, label='Netral (0)')
plt.fill_between(sentiment_trends.index, sentiment_trends.values, 0,
                 where=sentiment_trends.values > 0, color='mediumseagreen', alpha=0.2, label='Positif')
plt.fill_between(sentiment_trends.index, sentiment_trends.values, 0,
                 where=sentiment_trends.values < 0, color='tomato', alpha=0.2, label='Negatif')
plt.title('Tren Rata-rata Sentimen Ulasan Honest Review dari Waktu ke Waktu', fontsize=13)
plt.xlabel('Tanggal')
plt.ylabel('Rata-rata Polaritas Sentimen')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10) Word Cloud — Kata Paling Sering Muncul

**Word Cloud** memvisualisasikan frekuensi kata dalam seluruh teks ulasan — kata yang lebih besar muncul lebih sering. Berguna untuk mendapatkan gambaran cepat tentang topik utama ulasan pengguna.


In [ ]:
# Gabungkan semua teks menjadi satu string
text_data = ' '.join(df['content'].astype(str))

# Buat Word Cloud
wordcloud = WordCloud(
    width=900, height=450,
    background_color='white',
    max_words=200,
    colormap='viridis'
).generate(text_data)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud — Kata Paling Sering dalam Ulasan Honest Review', fontsize=14, pad=15)
plt.tight_layout()
plt.show()


## 11) Tampilkan Data Lengkap

Tampilkan DataFrame akhir yang sudah dilengkapi kolom analisis sentimen.

In [ ]:
print(f"Ringkasan hasil analisis sentimen:")
print(f"  Total ulasan         : {len(df)}")
print(f"  Rata-rata polaritas  : {df['sentiment_textblob'].mean():.4f}")
print(f"  Polaritas tertinggi  : {df['sentiment_textblob'].max():.4f}")
print(f"  Polaritas terendah   : {df['sentiment_textblob'].min():.4f}")
print()
df[['content', 'at', 'sentiment_textblob', 'sentiment_category']].head(10)
